## K8sGPT

Installieren Sie den K8sGPT-Operator mit Helm:


In [ ]:
%%bash
wget https://github.com/k8sgpt-ai/k8sgpt/releases/latest/download/k8sgpt_Linux_x86_64.tar.gz
tar -xzf k8sgpt_Linux_x86_64.tar.gz
sudo mv k8sgpt /usr/local/bin/

In [ ]:
%%bash
k8sgpt

---

### K8s Operator installieren 



In [ ]:
%%bash
helm upgrade --install k8sgpt-operator k8sgpt/k8sgpt-operator \
  -n k8sgpt-operator-system \
  --create-namespace \
  --set serviceMonitor.enabled=true \
  --set serviceMonitor.namespace=opentelemetry \
  --set grafanaDashboard.enabled=true \
  --set grafanaDashboard.namespace=opentelemetry

Erstellen Sie eine K8sGPT-Konfiguration mit Ihren KI-Anbietereinstellungen:

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: core.k8sgpt.ai/v1alpha1
kind: K8sGPT
metadata:
  name: k8sgpt
  namespace: k8sgpt-operator-system
spec:
  ai:
    enabled: false
    backend: openai
    model: gpt-4o-mini
  targetNamespace: yaml
  analysis:
    interval: 1m
  noCache: false
  version: v0.4.32
EOF

---

## Fehlerhaften Pod/Services starten

In [ ]:
%%bash
kubectl create namespace yaml
cat <<%EOF% | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  labels:
    app.kubernetes.io/name: webshop
  name: webshop
  namespace: yaml
spec:
  containers:
  - image: registry.gitlab.com/ch-mc-b/autoshop/shop:2.0.0
    name: webshop
%EOF%
cat <<%EOF% | kubectl apply -f -
apiVersion: v1
kind: Service
metadata:
  labels:
    app.kubernetes.io/name: webshop
  name: webshop
  namespace: yaml
spec:
  ports:
  - port: 8080
    protocol: TCP
    targetPort: 8080
  selector:
    app.kubernetes.io/name: webshop2
  type: LoadBalancer
%EOF%

In [ ]:
%%bash
k8sgpt analyze -n yaml -l german --explain       

---

### Fehlerkorrektur

---

## Dashboard (Grafana) und Metricen scrappen

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: k8sgpt-operator-metrics-reader-prometheus
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: k8sgpt-operator-metrics-reader
subjects:
  - kind: ServiceAccount
    name: prometheus-kube-prometheus-prometheus
    namespace: opentelemetry
EOF

In [ ]:
%%bash
NAMESPACE="opentelemetry"
SERVER_IP="$(cat ~/data/server-ip 2>/dev/null || true)"
PROMETHEUS_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-prometheus -o=jsonpath='{.spec.ports[?(@.port==9090)].nodePort}')"
GRAFANA_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-grafana -o=jsonpath='{.spec.ports[?(@.port==80)].nodePort}')"

echo "Grafana UI      : http://${SERVER_IP}:${GRAFANA_PORT}"
echo "Prometheus UI   : http://${SERVER_IP}:${PROMETHEUS_PORT}"

---

### Aufräumen


In [ ]:
%%bash
helm uninstall -n k8sgpt-operator-system k8sgpt-operator
